# 🎓 Capstone — Ship a real RAG service with Docker

You already know **RAG** and how to call an **LLM API**. What's new here is everything *around* the model: how to package a RAG app so it runs the same on any machine, and how to run its moving parts — a vector DB, the API, monitoring — together as **containers**.

So we'll build this capstone the way you'd actually build it at work, in three clear acts:

1. **Act 1 — Build the app and watch it work.** Stand up the vector DB, ingest some books, and run the RAG API *on your own machine* first. No image-building yet — just prove the logic.
2. **Act 2 — Package it as a Docker image.** Turn the working app into a small, reproducible image with a multi-stage `Dockerfile`.
3. **Act 3 — Orchestrate the whole stack with Compose.** Run the API + vector DB + Prometheus + Grafana together with a single command.

> **The golden rule for this notebook: one idea per step.** We introduce a piece, run it, see it work, *then* move on. Nothing is wired together until you've already met each part on its own.

### 🏗️ What we're building

```
   ┌─────────┐     POST /ask      ┌───────────────────────────┐   embeddings + chat   ┌──────────┐
   │ browser │ ─────────────────▶ │  api · Flask + gunicorn   │ ────────────────────▶ │  OpenAI  │
   │         │ ◀───────────────── │ /ask · /health · /metrics │ ◀──────────────────── │ (cloud)  │
   └─────────┘  answer + sources  └───────────────────────────┘                       └──────────┘
                          │ vector search                 ▲ Prometheus scrapes /metrics
                          ▼                                │
                    ┌──────────┐                    ┌────────────┐  ───▶  ┌──────────┐
                    │  chroma  │                    │ prometheus │        │ grafana  │
                    │  :8000   │                    │   :9090    │        │  :3000   │
                    └──────────┘                    └────────────┘        └──────────┘
                      vector DB                       metrics store         dashboards
```

Every box except **OpenAI** is a container we'll run. The browser only ever talks to the `api` service; `api` talks to **Chroma** for retrieval and to **OpenAI** for embeddings + answers.

### 📁 What we'll create (in the order we'll create it)

Everything lives in one folder, `rag_app/`. We don't write it all at once — each file appears in the act where we first need it:

```
rag_app/
  # Act 1 — build & run locally
  ingest.py              # download books -> chunk -> embed (OpenAI) -> store in Chroma
  app.py                 # Flask RAG API: /ask, /health, /metrics
  templates/index.html   # a tiny ask-a-question web UI
  # Act 2 — package as an image
  requirements.txt       # pinned dependencies
  Dockerfile             # multi-stage build (small final image)
  .dockerignore          # keep junk & secrets out of the image
  # Act 3 — orchestrate the stack
  compose.yaml           # api + chroma + prometheus + grafana, wired together
  prometheus.yml         # tells Prometheus to scrape the API
  .env.example           # template for your real .env (holds OPENAI_API_KEY)
```

Mental model: **app first → image → stack.** By the time we reach Compose, you'll already have run every piece by hand, so the Compose file is just "the same things, described once."

---
# 🟢 Act 1 — Build the app and watch it work

A RAG app has three moving parts: a place to **store** vectors, something to **embed** text, and something to **answer** questions. Two of those are just OpenAI API calls over the network. The only piece we have to *run* ourselves is the **vector database** — Chroma.

So Act 1 is short and concrete:

1. Start Chroma (your first container).
2. Ingest a few books into it (download → chunk → embed → store).
3. Run the Flask RAG API on your machine and ask it a question.

No `Dockerfile`, no Compose yet — we're just getting the app working.

## Step 1.1 — Start the vector database (your first container)

A **container** is just a process started from an **image** (a packaged filesystem + program). We don't have to install Chroma — we run its official image and Docker fetches it for us.

Run this in a terminal:

```bash
docker run -d \
  --name chroma \
  -p 8001:8000 \
  -v chroma_data:/data \
  -e IS_PERSISTENT=TRUE \
  -e PERSIST_DIRECTORY=/data \
  -e ANONYMIZED_TELEMETRY=FALSE \
  chromadb/chroma:1.0.15
```

What each flag does — this *is* the core of "Docker structure":

| Flag | Meaning |
|---|---|
| `-d` | **detached** — run in the background, give us our terminal back |
| `--name chroma` | a friendly name, so we can `docker logs chroma`, `docker rm chroma`, … |
| `-p 8001:8000` | **publish a port** (`HOST:CONTAINER`). Chroma listens on `8000` *inside*; we reach it at `localhost:8001` *outside* |
| `-v chroma_data:/data` | mount a **named volume** at `/data` so vectors survive even if the container is deleted |
| `-e KEY=VALUE` | set environment variables that configure the image |
| `chromadb/chroma:1.0.15` | the image, **pinned** to a version (never `:latest`) |

> We published Chroma on host port **8001** (not 8000) on purpose — in Act 3 our API will want host port 8000, so we keep Chroma out of its way from the start.

In [2]:
# Safe to run: is Chroma reachable on localhost:8001?
# (Friendly message if it is not up yet — start it with the `docker run` command above.)
import urllib.request

try:
    with urllib.request.urlopen("http://localhost:8001/api/v2/heartbeat", timeout=3) as r:
        print(f"✅ Chroma is up — heartbeat responded ({r.status}). Reachable at localhost:8001.")
except Exception as exc:
    print(f"⚠️  Chroma not reachable at localhost:8001 yet: {exc}")
    print("   Start it with the `docker run ... chromadb/chroma` command above, then re-run.")


✅ Chroma is up — heartbeat responded (200). Reachable at localhost:8001.


## Step 1.2 — Install the app's Python libraries (locally)

We'll run `ingest.py` and `app.py` **on this machine** first, talking to the Chroma container we just started. That means their libraries need to be available here. (In Act 2 we'll pin these exact versions into the image instead.)

You'll also need your OpenAI key available to these local runs:

```bash
export OPENAI_API_KEY=sk-...your-key...
```

In [ ]:
#%pip install -q openai==2.41.1 chromadb-client==1.0.15 requests==2.34.2 tqdm==4.68.2 flask==3.1.3 prometheus-flask-exporter==0.23.2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
deepteam 1.0.6 requires deepeval>=3.6.2, which is not installed.
tensorflow 2.18.1 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.3.3 which is incompatible.
litellm 1.83.4 requires openai==2.30.0, but you have openai 2.41.1 which is incompatible.
litellm 1.83.4 requires python-dotenv==1.0.1, but you have python-dotenv 1.2.2 which is incompatible.
streamlit 1.41.1 requires packaging<25,>=20, but you have packaging 25.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 1.3 — Ingest the books (download → chunk → embed → store)

This is the RAG ingestion you already know, pointed at our Chroma container as the store:

1. **Download** a handful of public-domain books from Project Gutenberg.
2. **Strip** the license boilerplate, keep the body.
3. **Chunk** each book into overlapping ~300-word windows (overlap keeps an idea from being split across a boundary).
4. **Embed** each chunk with OpenAI `text-embedding-3-small`.
5. **Store** the vectors + metadata (`title`, `doc_id`, `chunk_index`, `source`) in Chroma.

It's **idempotent**: if the collection already has vectors, re-running does nothing — so you only ever pay to embed once. First create the project folder, then write the script.

In [3]:
import os

# %%writefile won't create missing parent directories, so make them first.
os.makedirs("rag_app/templates", exist_ok=True)
print("Created rag_app/ and rag_app/templates/ — the %%writefile cells below fill them in.")

Created rag_app/ and rag_app/templates/ — the %%writefile cells below fill them in.


In [5]:
%%writefile rag_app/ingest.py
"""One-time corpus ingest: download a few Project Gutenberg books, chunk them,
embed with OpenAI, and add them to Chroma. Idempotent — re-running is a no-op once
the collection is populated (the "embed once" idea from the RAG demo notebook).

Run it once Chroma is reachable. Locally (Act 1), point it at the published port:
    CHROMA_HOST=localhost CHROMA_PORT=8001 python ingest.py
(or, against the Compose stack: docker compose run --rm api python ingest.py)
"""
import os
import re

import requests
import chromadb
from openai import OpenAI

try:
    from tqdm import tqdm
except ImportError:
    tqdm = None

CHROMA_HOST = os.environ.get("CHROMA_HOST", "localhost")
CHROMA_PORT = int(os.environ.get("CHROMA_PORT", "8000"))
COLLECTION  = os.environ.get("CHROMA_COLLECTION", "rag_demo")
EMBED_MODEL = os.environ.get("EMBED_MODEL", "text-embedding-3-small")

WORDS_PER_CHUNK = 300
OVERLAP_WORDS   = 60
EMBED_BATCH     = 64

# A small slice of the reference corpus — add more titles from the RAG notebook.
GUTENBERG_BOOKS = {
    "Moby-Dick": "https://www.gutenberg.org/files/2701/2701-0.txt",
    "Pride and Prejudice": "https://www.gutenberg.org/files/1342/1342-0.txt",
    "Frankenstein": "https://www.gutenberg.org/files/84/84-0.txt",
    "Alice in Wonderland": "https://www.gutenberg.org/cache/epub/11/pg11.txt",
    "Dracula": "https://www.gutenberg.org/files/345/345-0.txt",
    "A Tale of Two Cities": "https://www.gutenberg.org/files/98/98-0.txt",
    "The Great Gatsby": "https://www.gutenberg.org/cache/epub/64317/pg64317.txt",
    "Adventures of Sherlock Holmes": "https://www.gutenberg.org/files/1661/1661-0.txt",
    "War and Peace": "https://www.gutenberg.org/files/2600/2600-0.txt",
    "Jane Eyre": "https://www.gutenberg.org/files/1260/1260-0.txt",
    "The Picture of Dorian Gray": "https://www.gutenberg.org/files/174/174-0.txt",
    "Crime and Punishment": "https://www.gutenberg.org/files/2554/2554-0.txt",
    "Wuthering Heights": "https://www.gutenberg.org/files/768/768-0.txt"
}

oai = OpenAI()
chroma = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
# Chroma stores whatever vectors we give it; no embedding model lives in the DB.
collection = chroma.get_or_create_collection(
    COLLECTION, metadata={"hnsw:space": "cosine"}    # cosine distance: lower = closer
)

# Idempotency guard: if we already have vectors, do nothing.
if collection.count() > 0:
    print(f"'{COLLECTION}' already holds {collection.count()} vectors — nothing to do.")
    raise SystemExit(0)

# Project Gutenberg wraps each book in license boilerplate; keep only the body.
START_MARK = re.compile(r"\*\*\* START OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)
END_MARK   = re.compile(r"\*\*\* END OF (THIS|THE) PROJECT GUTENBERG EBOOK .* \*\*\*", re.I)

session = requests.Session()
session.trust_env = False            # don't route the download through a proxy/VPN

# 1) Download + 2) chunk each book into overlapping word windows.
chunks = []
for title, url in GUTENBERG_BOOKS.items():
    print(f"Downloading {title} ...")
    raw = session.get(url, timeout=180).text
    s, e = START_MARK.search(raw), END_MARK.search(raw)
    if s and e and e.start() > s.end():
        raw = raw[s.end():e.start()]

    doc_id = re.sub(r"[^a-z0-9]+", "-", title.lower()).strip("-")
    words = re.sub(r"\s+", " ", raw).strip().split()
    step = max(1, WORDS_PER_CHUNK - OVERLAP_WORDS)
    idx = 0
    for start in range(0, len(words), step):
        window = words[start:start + WORDS_PER_CHUNK]
        if len(window) < max(60, WORDS_PER_CHUNK // 4):
            break
        chunks.append({
            "id": f"{doc_id}#{idx}",           # Chroma takes string ids directly
            "doc_id": doc_id, "chunk_index": idx, "title": title,
            "source": url, "text": " ".join(window),
        })
        idx += 1
        if start + WORDS_PER_CHUNK >= len(words):
            break

print(f"Built {len(chunks)} chunks from {len(GUTENBERG_BOOKS)} books. Embedding with {EMBED_MODEL} ...")

# 3) Embed in batches and 4) add to Chroma (documents + metadata + our vectors).
spans = range(0, len(chunks), EMBED_BATCH)
for span in (tqdm(list(spans), desc="Embedding") if tqdm else spans):
    batch = chunks[span:span + EMBED_BATCH]
    vectors = [d.embedding for d in
               oai.embeddings.create(model=EMBED_MODEL, input=[c["text"] for c in batch]).data]
    collection.add(
        ids=[c["id"] for c in batch],
        embeddings=vectors,
        documents=[c["text"] for c in batch],
        metadatas=[{"doc_id": c["doc_id"], "chunk_index": c["chunk_index"],
                    "title": c["title"], "source": c["source"]} for c in batch],
    )

print(f"Done. '{COLLECTION}' now holds {collection.count()} vectors.")


Overwriting rag_app/ingest.py


### ▶️ Run the ingest

`ingest.py` reads its Chroma location from the environment. We published Chroma on `localhost:8001`, so point it there and run it (this calls the **paid** OpenAI embeddings API — a few cents for the whole corpus):

```bash
cd rag_app
CHROMA_HOST=localhost CHROMA_PORT=8001 python ingest.py
```

You'll see each book download, then a progress bar as it embeds. At the end:

```
Done. 'rag_demo' now holds 6231 vectors.
```

In [4]:
# Safe to run: how many vectors are in Chroma right now?
# (0 or an error means ingest has not run yet / Chroma is not up.)
try:
    import chromadb
    client = chromadb.HttpClient(host="localhost", port=8001)
    col = client.get_or_create_collection("rag_demo", metadata={"hnsw:space": "cosine"})
    print(f"✅ Collection 'rag_demo' holds {col.count()} vectors.")
except Exception as exc:
    print(f"⚠️  Could not read the collection yet: {exc}")
    print("   Make sure Chroma is up and `python ingest.py` has finished, then re-run.")


✅ Collection 'rag_demo' holds 8509 vectors.


## Step 1.4 — The RAG API (`app.py`)

Now the query side — a small **Flask** service that turns the retrieve-then-generate flow into HTTP endpoints:

- **`POST /ask`** — embed the question → ask Chroma for the top-K nearest chunks → stuff them into the prompt as context → call `gpt-4o-mini` with a strict *"answer only from this context, and cite your sources"* system prompt → return the answer + the source chunks.
- **`GET /health`** — is Chroma reachable and populated, and how many vectors? (We'll reuse this as a container health probe later.)
- **`GET /metrics`** — request counts + latencies in Prometheus format, added automatically by `prometheus-flask-exporter`, plus a custom `rag_questions_total` counter.

The OpenAI and Chroma clients are built **once at import** and reused on every request — you don't want to rebuild them per call.

In [13]:
%%writefile rag_app/app.py
"""Flask RAG service: retrieve from Chroma, answer with OpenAI, expose /metrics.

Wired to two backends via environment config (see compose.yaml / .env):
  - Chroma  (vector DB)             via CHROMA_HOST / CHROMA_PORT
  - OpenAI  (embeddings + chat LLM) via OPENAI_API_KEY
Prometheus scrapes /metrics; Grafana visualizes it.
"""
import os

from flask import Flask, request, jsonify, render_template
from prometheus_flask_exporter import PrometheusMetrics
from openai import OpenAI
import chromadb

# ---- Config: everything is overridable via the environment / .env ----
CHROMA_HOST = os.environ.get("CHROMA_HOST", "localhost")
CHROMA_PORT = int(os.environ.get("CHROMA_PORT", "8001"))
COLLECTION  = os.environ.get("CHROMA_COLLECTION", "rag_demo")
CHAT_MODEL  = os.environ.get("CHAT_MODEL", "gpt-4o-mini")
EMBED_MODEL = os.environ.get("EMBED_MODEL", "text-embedding-3-small")
TOP_K       = int(os.environ.get("TOP_K", "5"))

app = Flask(__name__)

# Auto-exposes /metrics with per-route request counts + latency histograms.
metrics = PrometheusMetrics(app)
metrics.info("rag_app_info", "RAG Flask service", version="1.0.0")

# Clients are built once at import and reused across requests.
oai = OpenAI()                       # reads OPENAI_API_KEY from the environment
chroma = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
# We always hand Chroma our own (OpenAI) vectors, so it needs no embedding model.
collection = chroma.get_or_create_collection(
    COLLECTION, metadata={"hnsw:space": "cosine"}   # cosine distance: lower = closer
)


def embed(text):
    """Turn one piece of text into an embedding vector via OpenAI."""
    resp = oai.embeddings.create(model=EMBED_MODEL, input=[text])
    return resp.data[0].embedding


def answer(question):
    """Retrieve the nearest chunks from Chroma, then ask the LLM to answer
    ONLY from that context (citing [doc_id#chunk] for each claim)."""
    res = collection.query(
        query_embeddings=[embed(question)], n_results=TOP_K,
        include=["documents", "metadatas", "distances"],
    )
    # We sent one query, so the results live at index [0].
    docs  = res["documents"][0]
    metas = res["metadatas"][0]
    dists = res["distances"][0]

    context = ""
    for text, meta in zip(docs, metas):
        context += f"[{meta['doc_id']}#{meta['chunk_index']}] {meta['title']}\n{text}\n---\n"

    system_prompt = (
        "You are a helpful assistant. Answer ONLY using the provided context. "
        "If the answer is not in the context, say: "
        "'I don't know based on the provided context.' "
        "Cite sources in square brackets like [doc_id#chunk_index] for each key claim."
    )
    resp = oai.chat.completions.create(
        model=CHAT_MODEL,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"Question: {question}\n\nContext:\n{context}"},
        ],
    )
    return resp.choices[0].message.content, metas, dists


@app.get("/")
def index():
    return render_template("index.html")


@app.get("/health")
def health():
    """Readiness probe: is Chroma reachable and the collection populated?"""
    try:
        return {"status": "ok", "vectors": collection.count()}
    except Exception as exc:
        return {"status": "degraded", "error": str(exc)}, 503


@app.post("/ask")
@metrics.counter("rag_questions_total", "Questions answered by the RAG API")
def ask():
    question = (request.get_json(silent=True) or {}).get("question", "").strip()
    if not question:
        return jsonify({"error": 'POST JSON like {"question": "..."}'}), 400
    try:
        text, metas, dists = answer(question)
    except Exception as exc:
        return jsonify({"error": str(exc)}), 502
    return jsonify({
        "question": question,
        "answer": text,
        "sources": [
            {"id": f"{m['doc_id']}#{m['chunk_index']}",
             "title": m["title"], "distance": round(d, 4)}
            for m, d in zip(metas, dists)
        ],
    })


if __name__ == "__main__":
    # Dev server only — the container runs gunicorn (see Dockerfile).
    app.run(host="0.0.0.0", port=8000)


Overwriting rag_app/app.py


In [8]:
%%writefile rag_app/templates/index.html
<!doctype html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>📚 Book RAG</title>
  <style>
    body { font-family: system-ui, sans-serif; max-width: 720px; margin: 40px auto; padding: 0 16px; }
    h1 { font-size: 1.4rem; }
    textarea { width: 100%; height: 72px; font-size: 1rem; padding: 8px; box-sizing: border-box; }
    button { margin-top: 8px; padding: 8px 18px; font-size: 1rem; cursor: pointer; }
    #answer { white-space: pre-wrap; background: #f6f8fa; padding: 16px; border-radius: 8px; margin-top: 16px; min-height: 1em; }
    .muted { color: #888; }
  </style>
</head>
<body>
  <h1>📚 Ask the library</h1>
  <p class="muted">Retrieval-augmented answers over a few classic books (Chroma + OpenAI).</p>
  <textarea id="q" placeholder="e.g. How does Victor Frankenstein create life?"></textarea>
  <button onclick="ask()">Ask</button>
  <div id="answer"></div>
  <script>
    async function ask() {
      const q = document.getElementById('q').value.trim();
      const out = document.getElementById('answer');
      if (!q) { out.textContent = 'Type a question first.'; return; }
      out.textContent = 'Thinking…';
      try {
        const r = await fetch('/ask', {
          method: 'POST',
          headers: { 'Content-Type': 'application/json' },
          body: JSON.stringify({ question: q })
        });
        const data = await r.json();
        if (data.error) { out.textContent = 'Error: ' + data.error; return; }
        const sources = (data.sources || [])
          .map(s => `• [${s.id}] ${s.title} (distance ${s.distance})`).join('\n');
        out.textContent = data.answer + '\n\n— sources —\n' + sources;
      } catch (e) {
        out.textContent = 'Request failed: ' + e;
      }
    }
  </script>
</body>
</html>


Overwriting rag_app/templates/index.html


### ▶️ Run the API and ask a question

Point the API at the same Chroma and start it (Flask's dev server listens on `8000`):

```bash
cd rag_app
CHROMA_HOST=localhost CHROMA_PORT=8001 python app.py
```

In another terminal, ask it something:

```bash
curl -s -X POST localhost:8000/ask \
  -H "Content-Type: application/json" \
  -d '{"question": "How does Victor Frankenstein create life?"}' | python -m json.tool
```

…or open **http://localhost:8000** and use the little UI. You should get a grounded answer with `sources` (each with a `distance` — lower = closer match). 🎉

**That's the entire RAG app working — with only one container running (Chroma).** Stop the dev server with `Ctrl-C` when you're done. Next we'll package this app into its *own* image so it no longer depends on what's installed on your machine.

---
# 📦 Act 2 — Package the app as a Docker image

Right now the app only runs because *your* machine has the right Python and libraries. To ship it to a teammate or a server — and to run it identically everywhere — we bake the app and its dependencies into an **image**.

Three files do this:

- `requirements.txt` — the exact library versions (reproducible builds),
- `Dockerfile` — the recipe for building the image,
- `.dockerignore` — what to leave out of the build.

## Step 2.1 — Pin the dependencies

We pin **exact** versions (`flask==3.1.3`, not just `flask`). Without pins, a rebuild months from now could pull newer, subtly-incompatible libraries — the opposite of "ships the same everywhere." These are the same libraries you installed in Act 1, now frozen.

In [5]:
%%writefile rag_app/requirements.txt
# Pin EXACT versions for reproducible builds (not just `flask`, etc.).
flask==3.1.3
gunicorn==26.0.0
openai==2.41.1
chromadb-client==1.0.15          # lightweight HTTP client (no server/onnx deps)
prometheus-flask-exporter==0.23.2
requests==2.34.2
tqdm==4.68.2


Overwriting rag_app/requirements.txt


## Step 2.2 — The multi-stage `Dockerfile`

A `Dockerfile` is a recipe: each line is a step that builds on the previous one. The trick here is **two stages**:

- **Stage 1 — `builder`:** starts from `python:3.12-slim`, installs compilers (`build-essential`), and builds all our dependencies into an isolated virtualenv at `/opt/venv`. This stage is bulky — but we throw it away.
- **Stage 2 — `runtime`:** starts fresh from a clean `python:3.12-slim` and copies **only the finished `/opt/venv`** from the builder. The compilers and caches never reach the final image, so it stays small (a few hundred MB — there's no `torch` here; the heavy lifting happens over the network at OpenAI).

It also folds in standard production hygiene:

- `PYTHONUNBUFFERED=1` so logs appear immediately;
- a **non-root** `appuser` (don't run as root);
- a **`HEALTHCHECK`** that hits `/health` (written in Python, since slim images have no `curl`);
- **gunicorn**, a real WSGI server, instead of Flask's dev server.

In [6]:
%%writefile rag_app/Dockerfile
# syntax=docker/dockerfile:1
# ============ STAGE 1: builder ============
FROM python:3.12-slim AS builder

ENV PIP_NO_CACHE_DIR=1
WORKDIR /app

# Build tools live ONLY in this stage and are never shipped to the runtime image.
RUN apt-get update && apt-get install -y --no-install-recommends \
        build-essential \
    && rm -rf /var/lib/apt/lists/*

# Isolated virtualenv we copy out wholesale into the runtime stage.
RUN python -m venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# ============ STAGE 2: runtime ============
FROM python:3.12-slim

ENV PYTHONUNBUFFERED=1 PYTHONDONTWRITEBYTECODE=1
RUN useradd --create-home --uid 1000 appuser
WORKDIR /app

# Copy ONLY the finished virtualenv from the builder — no compilers, no caches.
COPY --from=builder /opt/venv /opt/venv
ENV PATH="/opt/venv/bin:$PATH"

# App code (app.py, ingest.py, templates/), owned by the non-root user.
COPY --chown=appuser:appuser . .
USER appuser

EXPOSE 8000
# Liveness probe via Python (curl isn't in slim images).
HEALTHCHECK --interval=30s --timeout=5s --start-period=10s --retries=3 \
  CMD python -c "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')" || exit 1

# Production WSGI server. ONE worker keeps a single Prometheus metrics registry;
# threads give cheap concurrency for I/O-bound requests (each /ask waits on OpenAI
# + Chroma). Scale out with more workers + multiprocess metrics in production.
CMD ["gunicorn", "--bind", "0.0.0.0:8000", "--workers", "1", "--threads", "8", "--timeout", "120", "app:app"]


Overwriting rag_app/Dockerfile


## Step 2.3 — `.dockerignore`

Everything in the build folder is sent to Docker as the "build context." `.dockerignore` keeps that lean and — crucially — keeps secrets (`.env`) and junk (`__pycache__`, local data) **out of the image**.

In [ ]:
%%writefile rag_app/.dockerignore
# Keep the build context lean and secrets out of the image.
.git
.gitignore
__pycache__/
*.pyc
.venv/
venv/
.env
.env.*
*.ipynb_checkpoints
# Local artifacts that are rebuilt inside containers:
corpus/
chroma_storage/


### ▶️ Build the image

```bash
cd rag_app
docker build -t rag-app:1.0 .
```

Docker runs the `Dockerfile` top to bottom: builder stage, then runtime stage. Check the result:

```bash
docker images rag-app
# REPOSITORY   TAG   IMAGE ID       SIZE
# rag-app      1.0   ...            ~400MB
```

We *won't* `docker run` this image by itself — it needs Chroma alongside it, the right environment variables, and (soon) Prometheus + Grafana. Wiring several containers together is exactly what **Compose** is for. On to Act 3.

---
# 🟦 Act 3 — Orchestrate the whole stack with Compose

In Act 1 we started Chroma with one long `docker run`. The full application is **four** containers:

| Service | Image | Role |
|---|---|---|
| `api` | *we built it* | the Flask RAG service |
| `chroma` | `chromadb/chroma` | vector database |
| `prometheus` | `prom/prometheus` | scrapes the API's `/metrics` |
| `grafana` | `grafana/grafana` | dashboards on top of Prometheus |

Running four `docker run` commands by hand — each with its own ports, volumes, env, and start-order — and keeping them in sync is exactly the pain **Docker Compose** removes. One `compose.yaml` describes the whole stack; one command runs it.

**The one new idea:** inside a Compose stack, containers reach each other by **service name**, not `localhost`. So our API finds the database at **`chroma:8000`** (service name + the *container* port), not `localhost:8001`. That host `8001` mapping was only ever for *us*, on the host.

## Step 3.1 — Hand Chroma over to Compose

We started Chroma by hand; now Compose will manage its own Chroma container. Stop the standalone one to free the name and the host port:

```bash
docker rm -f chroma
```

Your embeddings are safe: they live in the **`chroma_data` named volume**, not in the container we just removed. In the Compose file we'll point Chroma at that *same* volume, so the vectors from Act 1 carry straight over — no re-embedding, no re-paying.

## Step 3.2 — Describe the stack in `compose.yaml`

One file, four services, all on a private network called `ragnet`. The parts worth reading closely:

- **`api`** — `build: .` builds the image from our `Dockerfile`. Its secret (`OPENAI_API_KEY`) comes from `.env` via `env_file` (injected at run time, never baked into the image); its non-secret config is set inline — note **`CHROMA_HOST=chroma`** (the service name!). `depends_on` + `condition: service_healthy` makes it wait until Chroma is actually ready.
- **`chroma`** — the pinned image, published on host `8001`, persisting to the **`chroma_data`** volume. We declare that volume **`external`** so Compose reuses the one you already filled in Act 1 instead of creating a brand-new empty one.
- **`prometheus`** — scrapes `api:8000/metrics` (config in `prometheus.yml`).
- **`grafana`** — reads from Prometheus; you build dashboards in its UI.

Named volumes (`chroma_data`, `grafana_data`) keep data alive across restarts.

In [11]:
%%writefile rag_app/compose.yaml
# The full RAG stack, wired together: Flask API + Chroma + Prometheus + Grafana.
# Bring it all up with:  docker compose up --build
# (No 'version:' key — it's obsolete in Compose v2.)

services:
  # ---------------- 1. The RAG API (Flask + gunicorn) ----------------
  api:
    build: .                          # the multi-stage Dockerfile in this dir
    ports:
      - "8000:8000"
    env_file:
      - .env                          # supplies OPENAI_API_KEY (never baked in)
      - /Users/shivam13juna/Documents/scaler/iitr_classes/llm_ref/openai_key.env
    environment:
      - CHROMA_HOST=chroma                # reach the DB by SERVICE NAME
      - CHROMA_PORT=8000
      - CHROMA_COLLECTION=rag_demo
      - CHAT_MODEL=gpt-5-nano
      - EMBED_MODEL=text-embedding-3-small
      - TOP_K=5
    depends_on:
      chroma:
        condition: service_healthy        # wait until Chroma answers its heartbeat
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8000/health')"]
      interval: 30s
      timeout: 5s
      retries: 3
      start_period: 10s
    restart: unless-stopped
    networks: [ragnet]

  # ---------------- 2. Vector database (Chroma) ----------------
  chroma:
    image: chromadb/chroma:1.0.15     # pinned, NOT :latest
    ports:
      # Chroma serves on 8000 inside the container; publish on 8001 to avoid
      # clashing with the api's 8000 on the host (host:container may differ).
      - "8001:8000"
    environment:
      - IS_PERSISTENT=TRUE            # write the index to disk...
      - PERSIST_DIRECTORY=/data       # ...here, which we back with a named volume
      - ANONYMIZED_TELEMETRY=FALSE
    volumes:
      - chroma_data:/data             # named volume => vectors persist
    healthcheck:
      test: ["CMD", "python", "-c", "import urllib.request; urllib.request.urlopen('http://localhost:8000/api/v2/heartbeat')"]
      interval: 10s
      timeout: 5s
      retries: 5
      start_period: 20s
    restart: unless-stopped
    networks: [ragnet]

  # ---------------- 3. Prometheus (scrapes /metrics) ----------------
  prometheus:
    image: prom/prometheus:v3.1.0
    ports:
      - "9090:9090"
    volumes:
      - ./prometheus.yml:/etc/prometheus/prometheus.yml:ro   # ':ro' = read-only
    depends_on: [api]
    networks: [ragnet]

  # ---------------- 4. Grafana (dashboards) ----------------
  grafana:
    image: grafana/grafana:11.4.0
    ports:
      - "3000:3000"
    environment:
      # Demo only — read this from a secret (Section 7) in real life.
      - GF_SECURITY_ADMIN_PASSWORD=admin
    volumes:
      - grafana_data:/var/lib/grafana
    depends_on: [prometheus]
    networks: [ragnet]

# Named volumes keep data alive across 'docker compose down' (use -v to wipe).
# chroma_data is EXTERNAL: reuse the volume we filled by hand in Act 1, so the
# embeddings carry over. Compose won't create it or delete it (even with -v).
volumes:
  chroma_data:
    external: true
  grafana_data:

# A private network for the stack; services find each other by name.
networks:
  ragnet:
    driver: bridge


Overwriting rag_app/compose.yaml


## Step 3.3 — Scrape config and secrets template

Two small files left: `prometheus.yml` tells Prometheus where to scrape, and `.env.example` documents the config — you copy it to `.env` and paste your real key there. The real `.env` is git-ignored and `.dockerignore`d, so the key never lands in version control or an image layer.

In [8]:
%%writefile rag_app/prometheus.yml
# Minimal Prometheus config: scrape the RAG API's /metrics endpoint.
global:
  scrape_interval: 15s

scrape_configs:
  - job_name: "rag-api"
    metrics_path: /metrics
    static_configs:
      # 'api' resolves via Compose DNS to the API container
      - targets: ["api:8000"]


Overwriting rag_app/prometheus.yml


In [9]:
%%writefile rag_app/.env.example
# Copy to .env and fill in your real key. NEVER commit the real .env.
# --- Secret (injected at run time, never baked into the image) ---
OPENAI_API_KEY=sk-replace-me

# --- Non-secret config (compose.yaml already sets sane defaults) ---
CHROMA_HOST=chroma
CHROMA_PORT=8000
CHROMA_COLLECTION=rag_demo
CHAT_MODEL=gpt-4o-mini
EMBED_MODEL=text-embedding-3-small
TOP_K=5


Overwriting rag_app/.env.example


### ✅ Validate the files before running anything

This is safe to run — it asks Compose to parse and type-check `compose.yaml` without starting a single container.

In [10]:
# Safe to run: validate the capstone Compose file (no containers started).
# compose's env_file needs a .env to exist, so seed a placeholder from the example.
import os, shutil
if not os.path.exists("rag_app/.env"):
    shutil.copy("rag_app/.env.example", "rag_app/.env")
    print("Seeded rag_app/.env from .env.example (placeholder key — edit before real use).")

!docker compose -f rag_app/compose.yaml config --quiet && echo "rag_app/compose.yaml is VALID ✅" || echo "compose.yaml has errors ❌"

Seeded rag_app/.env from .env.example (placeholder key — edit before real use).
rag_app/compose.yaml is VALID ✅


## Step 3.4 — Bring up the whole stack

```bash
cd rag_app
cp .env.example .env          # then edit .env and paste your real OPENAI_API_KEY

docker compose up --build -d
```

This builds the `api` image and starts all four containers in dependency order, waiting for Chroma's healthcheck before starting `api`. Because Compose reuses your `chroma_data` volume, Chroma already holds the Act-1 vectors — check:

```bash
curl -s localhost:8000/health        # {"status": "ok", "vectors": 6231}
```

> 🧩 **Need to (re)ingest into the running stack?** Two equivalent ways:
> - from your host, exactly like Act 1: `CHROMA_PORT=8001 python ingest.py`
> - the all-in-Docker way: `docker compose run --rm api python ingest.py` — this spins up a **throwaway** container from the `api` image, runs `ingest.py` *instead of* gunicorn, lets it reach `chroma` over `ragnet`, and (`--rm`) deletes it when done. Handy on a server with no local Python.

## Step 3.5 — Use the running service

```bash
# JSON API
curl -s -X POST localhost:8000/ask \
  -H "Content-Type: application/json" \
  -d '{"question": "Who is Mr. Darcy?"}' | python -m json.tool

# or the browser UI
open http://localhost:8000
```

Same app you ran in Act 1 — but now it's the containerized `api` image talking to the `chroma` service over the Compose network, not your local Python.

## Step 3.6 — See it monitored

Every `/ask` updates the metrics the API exposes at `/metrics`. Prometheus scrapes them; Grafana charts them.

```bash
open http://localhost:9090     # Prometheus — run the query:  rag_questions_total
open http://localhost:3000     # Grafana (login admin / admin)
```

In Grafana: add **Prometheus** as a data source (URL `http://prometheus:9090` — service name again!), then build a panel on `rag_questions_total` or the request-latency histograms. Ask a few questions and watch the numbers move.

## Step 3.7 — Tear it down

```bash
cd rag_app
docker compose down            # stop & remove the containers + network
```

`docker compose down` leaves your named volumes intact. Add `-v` to also wipe Grafana's data. Note that `chroma_data` is **external**, so Compose won't delete it even with `-v` — remove it deliberately with `docker volume rm chroma_data` if you want a truly clean slate.

---
# ✅ Recap — what you built

You took a RAG app from "runs on my machine" to "ships as a monitored, multi-container service," one linear step at a time:

| Act | You did | Docker idea it taught |
|---|---|---|
| **1** | Ran Chroma, ingested books, ran the API locally | images & containers, `-p` ports, `-v` named volumes, `-e` env |
| **2** | Wrote `requirements.txt`, `Dockerfile`, `.dockerignore`; built the image | multi-stage builds, pinning, non-root, healthchecks, `.dockerignore` |
| **3** | Wrote `compose.yaml` and ran the four-service stack | Compose, service-name networking, `depends_on`, secrets via `.env`, monitoring |

And the commands, in the order you ran them:

```bash
# Act 1 — build & run locally
docker run -d --name chroma -p 8001:8000 -v chroma_data:/data \
  -e IS_PERSISTENT=TRUE -e PERSIST_DIRECTORY=/data -e ANONYMIZED_TELEMETRY=FALSE \
  chromadb/chroma:1.0.15
CHROMA_HOST=localhost CHROMA_PORT=8001 python ingest.py
CHROMA_HOST=localhost CHROMA_PORT=8001 python app.py

# Act 2 — package
docker build -t rag-app:1.0 .

# Act 3 — orchestrate
docker rm -f chroma
docker compose up --build -d
docker compose down
```

That's a real, shareable AI application — **retrieval over a vector DB + an LLM, monitored, secured, and shipped as a lean multi-stage image** — built so that you understood every piece before they were wired together. 🚀